# 엑셀 기반 통계분석 자동화

**분석 순서:** 엑셀 불러오기 → 요인분석 → 변수계산 → 신뢰도 → 기술통계 → 상관관계 → 회귀분석 → 그래프

In [1]:
%config InlineBackend.figure_format = "retina"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ✅ 여기에 엑셀 파일명을 입력하세요
FILE = '데이터.xlsx'   # 예: '설문결과.xlsx'
SHEET = 0             # 첫 번째 시트 (이름으로도 가능: 'Sheet1')

df = pd.read_excel(FILE, sheet_name=SHEET)
print('✅ 데이터 로드 완료')
print(f'   행: {len(df)}개 | 열: {len(df.columns)}개')
print(f'   컬럼: {list(df.columns)}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '데이터.xlsx'

## 1단계: 요인분석 (Factor Analysis)

In [ ]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# ✅ 요인분석할 컬럼 목록 입력
FA_COLS = df.select_dtypes(include=[np.number]).columns.tolist()
# FA_COLS = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']  # 직접 지정할 경우

fa_data = df[FA_COLS].dropna()

# KMO 검정 (0.6 이상이면 요인분석 적합)
kmo_all, kmo_model = calculate_kmo(fa_data)
print(f'KMO 검정값: {kmo_model:.3f}  (0.6 이상이면 적합)')

# Bartlett 구형성 검정
chi2, p = calculate_bartlett_sphericity(fa_data)
print(f'Bartlett 검정: χ²={chi2:.3f}, p={p:.4f}  (p<0.05이면 유의)')

# ✅ 추출할 요인 수 설정
N_FACTORS = 3

fa = FactorAnalyzer(n_factors=N_FACTORS, rotation='varimax')
fa.fit(fa_data)

# 요인 부하량
loadings = pd.DataFrame(fa.loadings_, index=FA_COLS,
                        columns=[f'요인{i+1}' for i in range(N_FACTORS)])
print('\n📊 요인 부하량 (0.4 이상이면 유의):')
print(loadings.round(3))

# 고유값
ev, v = fa.get_eigenvalues()
print(f'\n고유값: {ev.round(3)}')

## 2단계: 변수계산 (평균 합산)

In [ ]:
# ✅ 각 요인(구성개념)별 문항 묶음 설정
FACTORS = {
    '요인A': ['Q1', 'Q2', 'Q3'],   # 예시 - 실제 컬럼명으로 변경
    '요인B': ['Q4', 'Q5', 'Q6'],
    '요인C': ['Q7', 'Q8', 'Q9'],
}

for name, cols in FACTORS.items():
    # 실제 존재하는 컬럼만 사용
    existing = [c for c in cols if c in df.columns]
    if existing:
        df[name] = df[existing].mean(axis=1)
        print(f'✅ {name} 생성 완료 (문항: {existing})')
    else:
        print(f'⚠️  {name}: 컬럼 없음 ({cols})')

print('\n변수계산 후 데이터 미리보기:')
df[list(FACTORS.keys())].describe().round(3)

## 3단계: 신뢰도 분석 (Cronbach's Alpha)

In [ ]:
import pingouin as pg

print('📊 신뢰도 분석 (Cronbach α)')
print('='*40)
print('기준: α ≥ 0.9 우수 | ≥ 0.8 양호 | ≥ 0.7 수용 | < 0.6 부적합')
print('='*40)

for name, cols in FACTORS.items():
    existing = [c for c in cols if c in df.columns]
    if len(existing) >= 2:
        alpha = pg.cronbach_alpha(data=df[existing])
        a_val = alpha[0]
        grade = '우수' if a_val>=0.9 else '양호' if a_val>=0.8 else '수용' if a_val>=0.7 else '부적합'
        print(f'{name}: α = {a_val:.3f}  → {grade}')

## 4단계: 기술통계분석

In [ ]:
# ✅ 분석할 변수 목록
VARS = list(FACTORS.keys())  # 위에서 만든 합산변수 사용
# VARS = ['변수1', '변수2']  # 직접 지정 가능

existing_vars = [v for v in VARS if v in df.columns]

desc = df[existing_vars].describe().T
desc['skewness'] = df[existing_vars].skew()
desc['kurtosis'] = df[existing_vars].kurtosis()
desc.columns = ['수', '평균', '표준편차', '최솟값', 'Q1', '중앙값', 'Q3', '최댓값', '왜도', '첨도']

print('📊 기술통계 결과:')
print(desc.round(3))

# 분포 시각화
fig, axes = plt.subplots(1, len(existing_vars), figsize=(5*len(existing_vars), 4))
if len(existing_vars) == 1:
    axes = [axes]
for ax, var in zip(axes, existing_vars):
    df[var].hist(ax=ax, bins=15, color='steelblue', edgecolor='white')
    ax.set_title(var)
    ax.set_xlabel('값')
plt.suptitle('변수별 분포', fontsize=14)
plt.tight_layout()
plt.show()

## 5단계: 상관관계분석

In [ ]:
corr = df[existing_vars].corr()

print('📊 상관관계 행렬:')
print(corr.round(3))

# 히트맵
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlBu_r',
            mask=mask, vmin=-1, vmax=1,
            annot_kws={'size': 12})
plt.title('상관관계 히트맵', fontsize=14)
plt.tight_layout()
plt.show()

# p값 포함 상세 출력
print('\n상관계수 유의성 (p < 0.05 → *유의):')
for i, v1 in enumerate(existing_vars):
    for v2 in existing_vars[i+1:]:
        result = pg.corr(df[v1], df[v2])
        r = result['r'].values[0]
        p = result['p-val'].values[0]
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
        print(f'  {v1} ↔ {v2}: r={r:.3f}, p={p:.4f} {sig}')

## 6단계: 회귀분석

In [ ]:
import statsmodels.api as sm

# ✅ 독립변수(X)와 종속변수(Y) 설정
X_VARS = existing_vars[:-1]   # 마지막 변수 제외 = 독립변수
Y_VAR  = existing_vars[-1]    # 마지막 변수 = 종속변수
# X_VARS = ['요인A', '요인B']  # 직접 지정 가능
# Y_VAR  = '요인C'

reg_data = df[X_VARS + [Y_VAR]].dropna()
X = sm.add_constant(reg_data[X_VARS])
Y = reg_data[Y_VAR]

model = sm.OLS(Y, X).fit()

print(f'📊 회귀분석: {Y_VAR} ~ {" + ".join(X_VARS)}')
print('='*60)
print(f'R²     = {model.rsquared:.3f}  (설명력)')
print(f'수정R² = {model.rsquared_adj:.3f}')
print(f'F통계량 = {model.fvalue:.3f}, p = {model.f_pvalue:.4f}')
print()
print('계수(기울기):')
coef_df = pd.DataFrame({
    '계수(β)': model.params,
    '표준오차': model.bse,
    't값': model.tvalues,
    'p값': model.pvalues,
    '유의': ['***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '' for p in model.pvalues]
})
print(coef_df.round(3))

## 7단계: 회귀 그래프 (기울기 시각화)

In [ ]:
fig, axes = plt.subplots(1, len(X_VARS), figsize=(6*len(X_VARS), 5))
if len(X_VARS) == 1:
    axes = [axes]

for ax, xvar in zip(axes, X_VARS):
    x = reg_data[xvar]
    y = reg_data[Y_VAR]

    # 산점도
    ax.scatter(x, y, alpha=0.5, color='steelblue', s=30)

    # 회귀선
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m*x_line + b, color='crimson', lw=2,
            label=f'기울기(β) = {m:.3f}')

    # 신뢰구간
    ax.fill_between(x_line,
                    m*x_line + b - 1.96*y.std()*np.sqrt(1/len(y)),
                    m*x_line + b + 1.96*y.std()*np.sqrt(1/len(y)),
                    alpha=0.15, color='crimson', label='95% 신뢰구간')

    ax.set_xlabel(xvar, fontsize=12)
    ax.set_ylabel(Y_VAR, fontsize=12)
    ax.set_title(f'{xvar} → {Y_VAR}', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('회귀분석 기울기 그래프', fontsize=15)
plt.tight_layout()
plt.savefig('회귀분석_결과.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 그래프 저장: 회귀분석_결과.png')